# 049 · SAM 2 carries each player's identity (Google Colab, GPU)

**hoop-reid-tracker**, players chapter, experiment 049 · results in
[`experiments/049_sam2_tracker/`](https://github.com/VaroLG/hoop-reid-tracker/tree/main/experiments/049_sam2_tracker).

The online tracker (011b) keeps identities with boxes, motion and appearance, and swaps them when players overlap.
Here a **SAM 2 video mask follows each player** frame to frame, and every box goes to the mask it contains, on the
same boxes as 011b. With 038 + 048 on top, on val (15 clips, every value chosen on train):

| | HOTA | AssA | IDF1 | IDSW |
|---|---|---|---|---|
| 011b + 038 + 048 (CPU pipeline) | 79.8 | 75.0 | 89.6 | 365 |
| **SAM 2 + 038 + 048** | **84.6** | **82.8** | **95.1** | **236** |

### How to use this notebook
* **Settings** (always): repository, packages, Drive, data. Run them top to bottom after every new session.
* **Part A · reproduce 049** (optional): every step skips what is already in your Drive, so a disconnected
  session resumes where it stopped.
* **Part B · SAM 2 identities for your clips or videos**: the identities only (steps 1-2 of the pipeline) on the
  GPU; the court, ball, possession and the video are made afterwards on the laptop with `run_pipeline.py`.

### Data, models and licences
| | |
|---|---|
| **Data** | SportsMOT basketball clips (**CC BY-NC 4.0**), downloaded here from Hugging Face with `scripts/download_data.py`. |
| **Your models, from Drive** | `Player_detector.pt` (011), `c06_best.pt` (C06, a copy of `outputs/court_runs/c06_yolo26n_seg/weights/best.pt`) and the folder `siglip_reid_sportsmot/` (028). They are yours and stay private (AGPL-3.0 + non-commercial data: not redistributed). |
| **SAM 2.1** | `facebook/sam2.1-hiera-small` (Meta, **Apache-2.0**), downloaded from Hugging Face by `transformers`. |
| **Results** | MOT files and scores in your Drive (`MyDrive/hooptrack/runs/049/`); not published. |

**Before running**: put those three in `MyDrive/hooptrack/weights/`. Runtime → change runtime type → **T4 GPU**.
`drive.mount` asks **you** to sign in and mounts **your** Drive.

## Settings

In [ ]:
# 1 · GPU and the repository: cloned the first time, updated (git pull) afterwards
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /content
!test -d /content/hoop-reid-tracker && git -C /content/hoop-reid-tracker pull -q || git clone -q https://github.com/VaroLG/hoop-reid-tracker
%cd /content/hoop-reid-tracker
!git log -1 --format="repository at %h · %s"

In [ ]:
# 2 · Only what this notebook needs, on top of Colab's own PyTorch (CUDA): never reinstall torch here
!pip -q install ultralytics==8.4.182 motmetrics==1.4.0 transformers==5.21.0 huggingface_hub==2.2.0 \
    "trackeval @ git+https://github.com/JonathonLuiten/TrackEval.git@12c8791b303e0a0b50f753af204249e622d0281a"
!pip -q install -e . --no-deps
import torch, transformers
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "| transformers", transformers.__version__)
from transformers import Sam2VideoModel  # noqa: F401  (fails here if this transformers has no SAM 2 video)

In [ ]:
# 3 · Drive, the models, the chosen values and the helpers used by every cell below
import json, os, shutil, subprocess, zipfile
from pathlib import Path
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/hooptrack")
WEIGHTS = DRIVE / "weights"
RUNS = DRIVE / "runs" / "049"; RUNS.mkdir(parents=True, exist_ok=True)
DETECTOR, COURT_MASK, SIGLIP = WEIGHTS / "Player_detector.pt", WEIGHTS / "c06_best.pt", WEIGHTS / "siglip_reid_sportsmot"
for p in (DETECTOR, COURT_MASK, SIGLIP):
    assert p.exists(), f"missing {p}: copy it to your Drive first"

# the repository's default paths point at these models (configs/player.yaml, run_pipeline.py): link them there
for src, dst in ((DETECTOR, "weights/Player_detector.pt"), (SIGLIP, "weights/siglip_reid_sportsmot"),
                 (COURT_MASK, "outputs/court_runs/c06_yolo26n_seg/weights/best.pt")):
    dst = Path(dst); dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        dst.symlink_to(src)

TRAIN_CLIPS = ["v_-6Os86HzwCs_c009", "v_2j7kLB-vEEk_c010", "v_4LXTUim5anY_c013"]   # held out: values chosen here
MODEL = "facebook/sam2.1-hiera-small"
DTYPE = "float16"          # T4: float16 runs on its tensor cores (bfloat16 is only emulated there); "auto" on A100 / L4
CHOSEN = (120, 0.4, 0.3)   # window (frames), min share of a mask inside a box, keep bonus: chosen on train (049)

def sh(cmd):
    """Run a command, print its output as it comes, stop the notebook if it fails (end of the output in the error)."""
    print("$", cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end=""); tail = (tail + [line])[-40:]
    assert p.wait() == 0, f"failed: {cmd}\n" + "".join(tail)

def done(run, clips):
    """A run is done when every clip has its results file (either layout of the repository)."""
    return all((run / "results" / f"{c}.txt").is_file() or (run / c / "results" / f"{c}.txt").is_file() for c in clips)

def track(split, clips, out):
    """1 · the online tracker 011b: the baseline, and the boxes SAM 2 works on."""
    if done(out, clips):
        print(out.name, "done"); return out
    only = "[" + ",".join(clips) + "]"
    sh(f'python -m hooptrack.cli track --config configs/player.yaml --set data.split={split} "data.only={only}" '
       f"detector.device=cuda reid.device=cuda filters.segmentation.device=cuda output.dir={out}")
    return out

def tag(w, s, k):
    return f"sam2_w{w}_s{s}" + (f"_k{k}" if k else "")      # k = 0 keeps the names of the runs made before inertia

def sam2(split, clips, run_in, out, w, s, k):
    """SAM 2 on the boxes of an online run (scripts/sam2_track.py skips the clips already done)."""
    if done(out, clips):
        print(out.name, "done"); return out
    sh(f"python scripts/sam2_track.py --run {run_in} --set data.split={split} --clips {' '.join(clips)} "
       f"--model {MODEL} --window {w} --min-share {s} --keep-bonus {k} --dtype {DTYPE} --out {out}")
    return out

def nested(run, clips):
    """<run>/results/<clip>.txt -> <run>_nested/<clip>/results/<clip>.txt, the layout 048 reads."""
    dst = run.with_name(run.name + "_nested")
    for c in clips:
        d = dst / c / "results"; d.mkdir(parents=True, exist_ok=True)
        shutil.copy(run / "results" / f"{c}.txt", d / f"{c}.txt")
    return dst

def final(split, clips, run_in, out, closed=True):
    """038 (cut at every crossing, pieces given to exactly 10 players by SigLIP) and 048 (body smoothing).
    closed=False: 048 only, on the tracker's own identities."""
    if done(Path(f"{out}_048"), clips):
        print(Path(f"{out}_048").name, "done"); return Path(f"{out}_048")
    ids = Path(f"{out}_038")
    if closed:
        sh(f"python scripts/stitch_tracklets.py --run {run_in} --config configs/player.yaml --clips {' '.join(clips)} "
           f"--set data.split={split} reid.type=siglip reid.weights={SIGLIP} reid.device=cuda "
           f"--split-crossings --closed-set 10 --continuity 0.3 --out {ids}")
    else:
        ids = nested(run_in, clips)
    sh(f"python scripts/smooth_body_labels.py --run {ids} --out {out}_048")
    return Path(f"{out}_048")

def hota(split, runs, csv):
    """HOTA / AssA / IDF1 / MOTA / IDSW of each run, against the ground truth (scripts/eval_hota.py)."""
    pairs = " ".join(f"{k}={v}" for k, v in runs.items())
    sh(f"python scripts/eval_hota.py --config configs/player.yaml --set data.split={split} --runs {pairs} --out {csv}")
    t = pd.read_csv(csv)
    return t[t.sequence == "OVERALL"].set_index("run")[["HOTA", "AssA", "IDF1", "MOTA", "IDSW"]]
print("settings ready")

In [ ]:
# 4 · Data: SportsMOT, basketball clips only (if Hugging Face asks for a token: a Colab secret named HF_TOKEN)
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except Exception:
    pass
if not Path("data/SportsMOT/dataset/val").is_dir():
    sh("python scripts/download_data.py --splits train val")
VAL_CLIPS = sorted(p.name for p in Path("data/SportsMOT/dataset/val").iterdir())
print(len(VAL_CLIPS), "val clips,", sum(Path(f"data/SportsMOT/dataset/train/{c}").is_dir() for c in TRAIN_CLIPS),
      "of 3 held-out train clips")

---
# Part A · Reproduce 049
Every step reuses what is in your Drive. Values are chosen on the 3 held-out **train** clips; val runs **once**.

In [ ]:
# A1 · The online tracker 011b on train and val (val: ~1 h on a T4)
track("train", TRAIN_CLIPS, RUNS / "011b_train")
track("val", VAL_CLIPS, RUNS / "011b_val")

In [ ]:
# A2 · SAM 2 on train: window x min share, then the inertia (keep bonus) at the best window. ~25 min per setting.
# FULL = False runs only the settings of the final comparison (no inertia vs the chosen inertia).
FULL = False
GRID = ([(w, s, 0.0) for w in (15, 30) for s in (0.4, 0.6)] + [(60, 0.4, 0.0), (240, 0.4, 0.0)] if FULL else []) \
    + [(120, 0.4, 0.0), (120, 0.4, 0.3), (120, 0.4, 0.6)]
runs, params = {"011b": RUNS / "011b_train"}, {}
for w, s, k in GRID:
    runs[tag(w, s, k)] = sam2("train", TRAIN_CLIPS, RUNS / "011b_train", RUNS / tag(w, s, k).replace("sam2_", "sam2_train_"), w, s, k)
    params[tag(w, s, k)] = (w, s, k)
train_table = hota("train", runs, RUNS / "hota_train.csv")
best = train_table.drop(index="011b")["HOTA"].idxmax()
print("best on train:", best, params[best], "| used below:", CHOSEN)
train_table

In [ ]:
# A3 · Optional diagnostic: how many identity switches come back to the previous id within 10 frames (A -> B -> A)
import sys; sys.path.insert(0, "scripts")
from render_compare import switch_records
from hooptrack.data.sportsmot import read_seqinfo
from hooptrack.eval.gt_teams import player_teams
from hooptrack.data.sportsmot import load_mot_txt

def flicker(run):
    for c in TRAIN_CLIPS:
        seq = read_seqinfo(Path(f"data/SportsMOT/dataset/train/{c}"))
        recs = switch_records(load_mot_txt(run / "results" / f"{c}.txt"), seq.load_gt(), seq.length, player_teams(seq))
        back = sum(any(q["gt_id"] == r["gt_id"] and 0 < q["frame"] - r["frame"] <= 10
                       and q["run_id"] == r["previous_run_id"] for q in recs) for r in recs)
        print(f"{run.name} {c}: {len(recs)} switches, {back} come back within 10 frames")

flicker(RUNS / "sam2_train_w120_s0.4")        # no inertia: most switches are flicker in piles
flicker(RUNS / "sam2_train_w120_s0.4_k0.3")   # the chosen inertia

In [ ]:
# A4 · 038 + 048 on top, on train: is 038 still needed on top of SAM 2? (a few minutes)
SAM_TRAIN = RUNS / tag(*CHOSEN).replace("sam2_", "sam2_train_")
top_train = {"011b+038+048": final("train", TRAIN_CLIPS, RUNS / "011b_train", RUNS / "011b_train"),
             "sam2+038+048": final("train", TRAIN_CLIPS, SAM_TRAIN, RUNS / "sam2_train_final"),
             "sam2+048":     final("train", TRAIN_CLIPS, SAM_TRAIN, RUNS / "sam2_train_only048", closed=False)}
t = hota("train", top_train, RUNS / "hota_train_final.csv")
CLOSED = t.drop(index="011b+038+048")["HOTA"].idxmax() == "sam2+038+048"
(RUNS / "chosen_on_train.json").write_text(json.dumps(dict(zip(("window", "min_share", "keep_bonus"), CHOSEN),
                                                               closed_set_on_top=bool(CLOSED))))
print("chosen on train: 038 on top of SAM 2 =", CLOSED)
t

In [ ]:
# A5 · Val, once, with what was chosen on train (SAM 2 on 15 clips: ~3.5 h on a T4; then a few minutes)
sam2("val", VAL_CLIPS, RUNS / "011b_val", RUNS / "sam2_val", *CHOSEN)
display(hota("val", {"011b": RUNS / "011b_val", "sam2": RUNS / "sam2_val"}, RUNS / "hota_val.csv"))
top = {"011b+038+048": final("val", VAL_CLIPS, RUNS / "011b_val", RUNS / "011b_val"),
       "sam2+038+048": final("val", VAL_CLIPS, RUNS / "sam2_val", RUNS / "sam2_val", closed=CLOSED)}
hota("val", top, RUNS / "hota_val_final.csv")

In [ ]:
# A6 · Download the scores and the val identities (for experiments/049_sam2_tracker/ and the laptop's videos)
from google.colab import files
with zipfile.ZipFile("/content/049_results.zip", "w") as z:
    for f in RUNS.glob("hota_*.csv"):
        z.write(f, f"049/{f.name}")
    z.write(RUNS / "chosen_on_train.json", "049/chosen_on_train.json")
    for f in (RUNS / "sam2_val_048").rglob("*.txt"):
        z.write(f, f"049/sam2_val_048/{f.relative_to(RUNS / 'sam2_val_048')}")
files.download("/content/049_results.zip")

On the laptop: unzip into `outputs/runs/` (it makes `outputs/runs/049/`), copy the `.csv` / `.json` to
`experiments/049_sam2_tracker/`, and draw any val clip with SAM 2's identities (court, ball, possession, video):

```
.venv\Scripts\python scripts/run_pipeline.py --clip v_BgwzTUxJaeU_c014 --court NCAA ^
    --players-from outputs/runs/049/sam2_val_048 --players-label sam2
```

---
# Part B · SAM 2 identities for your clips or videos
`run_pipeline.py --tracker sam2 --players-only`: the online tracker, SAM 2 (the values chosen in 049), 038 and 048,
and nothing else. Give SportsMOT clip names, or video files you put in `MyDrive/hooptrack/videos/`. ~1 fps.

In [ ]:
# B1 · What to run: SportsMOT clips (any split you downloaded) and / or videos in your Drive (name -> file, seconds)
CLIPS = []    # SportsMOT clip names, e.g. ["v_BgwzTUxJaeU_c014"]
# name -> (file in MyDrive/hooptrack/videos/, first second, last second or None). Use the SAME name and seconds as on
# the laptop, so both cut the same frames: the Mamba clips were run whole, as mamba_clip1 ... mamba_clip5.
VIDEOS = {f"mamba_clip{i}": (f"Clip{i}.mp4", 0, None) for i in range(1, 6)}
OUT = DRIVE / "pipeline_sam2"

for c in CLIPS:
    sh(f"python scripts/run_pipeline.py --clip {c} --tracker sam2 --players-only --device cuda --out {OUT}")
for name, (file, t0, t1) in VIDEOS.items():
    span = f"--start {t0}" + (f" --end {t1}" if t1 is not None else "")
    sh(f"python scripts/run_pipeline.py --video {DRIVE / 'videos' / file} --name {name} {span} "
       f"--tracker sam2 --players-only --device cuda --out {OUT}")

In [ ]:
# B2 · Download the identities: sam2_identities/<clip>/results/<clip>.txt
from google.colab import files
with zipfile.ZipFile("/content/sam2_identities.zip", "w") as z:
    for name in CLIPS + list(VIDEOS):
        for f in (OUT / name).glob("players__sam2/*/results/*.txt"):
            z.write(f, f"sam2_identities/{name}/results/{f.name}")
files.download("/content/sam2_identities.zip")

On the laptop: unzip into `outputs/runs/` and run the rest of the pipeline with those identities (for a video, the
same file, name and seconds as in B1, so the frames are the same):

```
.venv\Scripts\python scripts/run_pipeline.py --clip v_BgwzTUxJaeU_c014 --court NCAA ^
    --players-from outputs/runs/sam2_identities --players-label sam2

.venv\Scripts\python scripts/run_pipeline.py --video data/Mamba_games/Clip3.mp4 --name mamba_clip3 ^
    --venue mamba --players-from outputs/runs/sam2_identities --players-label sam2
```
The videos and metrics end in `__sam2` (`outputs/pipeline/<clip>/`), next to the CPU pipeline's.